<a href="https://colab.research.google.com/github/AndresVillotaVillota/AndresVillotaVillota-Competencia-Kaggle-2025-2/blob/main/02_preprocesado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import zipfile, os

# Subir el archivo ZIP (ejemplo: udea-ai-4-eng-20251-pruebas-saber-pro-colombia.zip)
uploaded = files.upload()

# Extraer ZIP
zip_name = list(uploaded.keys())[0]
extract_path = "datos_saberpro"

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Archivos extraídos:", os.listdir(extract_path))

Saving udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip to udea-ai-4-eng-20252-pruebas-saber-pro-colombia (2).zip
✅ Archivos extraídos: ['train.csv', 'submission_example.csv', 'test.csv']


In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv("datos_saberpro/train.csv")
test = pd.read_csv("datos_saberpro/test.csv")
sample_submission = pd.read_csv("datos_saberpro/submission_example.csv")

print("✅ Train:", train.shape, " Test:", test.shape)
train.head()

✅ Train: (692500, 21)  Test: (296786, 20)


,ID,PERIODO_ACADEMICO,E_PRGM_ACADEMICO,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_EDUCACIONPADRE,F_TIENELAVADORA,...,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,F_TIENECOMPUTADOR,F_TIENEINTERNET.1,F_EDUCACIONMADRE,RENDIMIENTO_GLOBAL,INDICADOR_1,INDICADOR_2,INDICADOR_3,INDICADOR_4
0,904256,20212,ENFERMERIA,BOGOTÁ,Entre 5.5 millones y menos de 7 millones,Menos de 10 horas,Estrato 3,Si,Técnica o tecnológica incompleta,Si,...,N,No,Si,Si,Postgrado,medio-alto,0.322,0.208,0.310,0.267
1,645256,20212,DERECHO,ATLANTICO,Entre 2.5 millones y menos de 4 millones,0,Estrato 3,No,Técnica o tecnológica completa,Si,...,N,No,Si,No,Técnica o tecnológica incompleta,bajo,0.311,0.215,0.292,0.264
2,308367,20203,MERCADEO Y PUBLICIDAD,BOGOTÁ,Entre 2.5 millones y menos de 4 millones,Más de 30 horas,Estrato 3,Si,Secundaria (Bachillerato) completa,Si,...,N,No,No,Si,Secundaria (Bachillerato) completa,bajo,0.297,0.214,0.305,0.264
3,470353,20195,ADMINISTRACION DE EMPRESAS,SANTANDER,Entre 4 millones y menos de 5.5 millones,0,Estrato 4,Si,No sabe,Si,...,N,No,Si,Si,Secundaria (Bachillerato) completa,alto,0.485,0.172,0.252,0.190
4,989032,20212,PSICOLOGIA,ANTIOQUIA,Entre 2.5 millones y menos de 4 millones,Entre 21 y 30 horas,Estrato 3,Si,Primaria completa,Si,...,N,No,Si,Si,Primaria completa,medio-bajo,0.316,0.232,0.285,0.294


In [3]:
train["source"] = "train"
test["source"] = "test"
test["RENDIMIENTO_GLOBAL"] = np.nan

data = pd.concat([train, test], ignore_index=True)

# Imputar faltantes
for col in data.select_dtypes(include="number"):
    data[col] = data[col].fillna(data[col].median())

for col in data.select_dtypes(include="object"):
    data[col] = data[col].fillna("missing")

print("✅ Datos combinados:", data.shape)

✅ Datos combinados: (989286, 22)


In [4]:
num_cols = data.select_dtypes(include=["number"]).columns.tolist()
cat_cols = data.select_dtypes(exclude=["number"]).drop(["source", "RENDIMIENTO_GLOBAL"], axis=1).columns.tolist()

print("🔢 Numéricas:", len(num_cols))
print("🔤 Categóricas:", len(cat_cols))

🔢 Numéricas: 6
🔤 Categóricas: 14


In [5]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Pipelines por tipo de dato
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Ensamblar pipeline completo
full_pipeline = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

print("✅ full_pipeline creado correctamente.")

✅ full_pipeline creado correctamente.


In [6]:
train_clean = data[data["source"] == "train"].drop(columns=["source"])
test_clean = data[data["source"] == "test"].drop(columns=["source", "RENDIMIENTO_GLOBAL"])

X_train = train_clean.drop(columns=["RENDIMIENTO_GLOBAL"])
y_train = train_clean["RENDIMIENTO_GLOBAL"]
X_test = test_clean.copy()

# Aplicar pipeline
X_train_transformed = full_pipeline.fit_transform(X_train)
X_test_transformed = full_pipeline.transform(X_test)

print("✅ Transformación completada:")
print("Train:", X_train_transformed.shape, " Test:", X_test_transformed.shape)

✅ Transformación completada:
Train: (692500, 1054)  Test: (296786, 1054)


In [7]:
import joblib, json

out_dir = "procesados"
os.makedirs(out_dir, exist_ok=True)

# Guardar datasets procesados
pd.DataFrame(X_train_transformed).to_csv(os.path.join(out_dir, "X_train_scaled.csv"), index=False)
pd.DataFrame(X_test_transformed).to_csv(os.path.join(out_dir, "X_test_scaled.csv"), index=False)
y_train.to_csv(os.path.join(out_dir, "y_train.csv"), index=False)

# Guardar pipeline
joblib.dump(full_pipeline, os.path.join(out_dir, "pipeline.joblib"))

# Crear metadatos seguros para JSON
meta = {
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "train_shape": X_train_transformed.shape,
    "test_shape": X_test_transformed.shape
}

with open(os.path.join(out_dir, "preprocessing_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print("✅ Archivos guardados en:", out_dir)

✅ Archivos guardados en: procesados


In [8]:
import shutil
from google.colab import files

# Comprimir toda la carpeta en un ZIP
shutil.make_archive("procesados", "zip", "procesados")

# Descargar el archivo ZIP con todo el contenido
files.download("procesados.zip")

print("✅ Archivo ZIP listo para descargar.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Archivo ZIP listo para descargar.


In [ ]:
from google.colab import files
files.download("procesados.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>